# Reproducibility and Inspection

Run yesterday's experiment again and the loss curve comes out different.
Is this morning's change an improvement, or a lucky seed? Answering that
question requires knowing where every random number in a training run comes
from. And once two runs *do* disagree, or a loss turns into NaN, the next
question is what the model computed layer by layer, ideally without editing
its source. This section covers both skills: controlling randomness (seeds,
generators, determinism) and observing a running model from the outside
(hooks).

In [1]:
import tensorflow as tf

## Seeds and Randomness

Randomness enters a training run in more places than you might list on a
first try: initialization draws every weight (that section),
dropout samples a fresh mask at each step
[@Srivastava.Hinton.Krizhevsky.ea.2014], the data loader shuffles
examples differently in every epoch, augmentations sample crops and flips,
and the loader's worker processes each do all of this in parallel. Every one
of these consults a pseudorandom number generator, a deterministic algorithm
whose entire output sequence is fixed by its seed. Seed every generator and
the run is repeatable; miss one and it is not.

TensorFlow collapses the inventory to one call:
`tf.keras.utils.set_random_seed(k)` seeds Python's `random` module, NumPy's
global generator, and TensorFlow's global generator together, the three a
typical training script consults. It does not reach generators that carry
their own state (`tf.random.Generator` objects, NumPy `Generator` objects
from `np.random.default_rng`), each of which has its own seed. The function
below draws everything, weights and data alike, from the seeded global
state, so one call pins the whole run:

In [2]:
def train_once(seed):
    tf.keras.utils.set_random_seed(seed)  # seeds Python, NumPy, and TF
    with tf.device('/CPU:0'):  # GPU kernels are not bitwise-deterministic
        net = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1)])
        opt = tf.keras.optimizers.SGD(learning_rate=0.1)
        X, y = tf.random.normal((128, 20)), tf.random.normal((128, 1))
        for _ in range(5):
            with tf.GradientTape() as tape:
                loss = tf.reduce_mean((net(X) - y) ** 2)
            grads = tape.gradient(loss, net.trainable_variables)
            opt.apply_gradients(zip(grads, net.trainable_variables))
        return float(loss), tf.identity(net.layers[0].kernel)

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      loss_a == loss_b, bool(tf.reduce_all(w_a == w_b)))
print('different seed:', loss_a, 'vs', loss_c)

same seed, identical loss and weights: True True
different seed: 1.0948041677474976 vs 1.0242893695831299


The two seeded runs agree *bitwise*, down to the last floating-point bit:
same initial weights, same data, same gradients, same operations in the
same order. One detail deserves a flag: we pinned the TensorFlow run to
the CPU. On a GPU the same seeded code produces equal losses but weights
that differ in their last bits — the reason emerges in the next section.

### Generator Objects

A single global stream is fragile: every consumer shares it, so inserting
one extra random call shifts everything drawn after it. A
`tf.random.Generator` is a private stream with its own seed. Here a data
split stays fixed no matter what else consumes randomness in between (the
generator has no permutation method, so we sort random uniforms, the
standard trick):

In [3]:
g = tf.random.Generator.from_seed(42)
split = tf.argsort(g.uniform((10,)))  # a random permutation from g's stream
_ = tf.random.normal((1000,))  # unrelated consumption of the global stream
split_again = tf.argsort(tf.random.Generator.from_seed(42).uniform((10,)))
print(bool(tf.reduce_all(split == split_again)), split.numpy())

True [4 2 7 3 5 8 9 1 0 6]


Sampling methods live on the generator itself (`g.normal`, `g.uniform`), so
a consumer that should own its randomness takes a `Generator` argument. The
input pipeline instead takes a seed directly, which we use next.

### DataLoader Workers

The classic reproducibility hole in the loader-worker world is that
parallel workers inherit or reseed a hidden generator: on fork-based
loaders every child starts with a byte-for-byte copy of the parent's NumPy
state, so all workers apply the same "random" augmentations, or each child
seeds itself from entropy and no two runs agree. `tf.data` sidesteps the
bug class by construction: the pipeline parallelizes with threads inside
one process, so there is no forked child to inherit a stale copy, and its
randomness is an explicit argument, `Dataset.shuffle(buffer, seed=...)`,
not an ambient global that a seeding call may or may not reach. What
remains is choosing what the seed means across epochs. With the default
`reshuffle_each_iteration=True`, a seeded pipeline produces a *different*
order on each pass but the same *sequence* of orders every time the
pipeline is rebuilt, fresh shuffles per epoch, repeatable across runs:

In [4]:
ds = tf.data.Dataset.range(8).shuffle(8, seed=0).batch(4)
print([b.numpy().tolist() for b in ds])  # epoch 1
print([b.numpy().tolist() for b in ds])  # epoch 2: reshuffled, still seeded
ds = tf.data.Dataset.range(8).shuffle(8, seed=0).batch(4)
print([b.numpy().tolist() for b in ds])  # rebuilt pipeline: epoch 1 again

[[5, 6, 7, 2], [1, 3, 4, 0]]
[[4, 6, 7, 2], [0, 5, 3, 1]]
[[5, 6, 7, 2], [1, 3, 4, 0]]


Both alternatives are explicit choices rather than accidents:
`reshuffle_each_iteration=False` freezes one order for every epoch, and
leaving `seed` unset draws fresh orders each run. One knob remains that
trades reproducibility away on purpose: parallel `map` and `interleave`
accept `deterministic=False`, which hands elements on in completion order
for speed; the default `True` keeps the pipeline's output order fixed.

### Randomness as a Value

Every failure above is hidden global state: a generator that lives
somewhere your code does not mention, silently copied or shared. Some
library designs remove the hiding place altogether. In an explicit-PRNG
design (JAX is the prominent example), randomness is threaded through the
program as a value: every random operation takes a key argument, using the
same key twice yields the same numbers by definition, and independent
streams are obtained by explicitly splitting a key in two. Accidentally
duplicating a stream would require writing the duplication into the code,
so the worker bug cannot be expressed. The price is bookkeeping, since
every function that needs randomness must accept and split keys. PyTorch's
global seed is the convenient version of the same contract, one implicit
key that everything shares; generator objects recover the explicit version
where it matters.

TensorFlow ships both designs side by side: the stateful API above (a
global generator plus `tf.random.Generator` objects) and a stateless
family in which the key is an argument:
`tf.random.stateless_normal(shape, seed=[k1, k2])` returns the same
numbers for the same seed pair by definition. The explicit `seed=`
arguments of `tf.data` are the same idea applied to the input pipeline.

## Determinism and Its Price

Seeding fixes which numbers the program draws. It does not fix how the
arithmetic evaluates. Floating-point addition is not associative
(that section), so summing the same numbers in a different
order or grouping may give a different answer:

In [5]:
x = tf.random.Generator.from_seed(0).normal((1_000_000,))
s_fwd = tf.reduce_sum(x)
s_rev = tf.reduce_sum(x[::-1])  # same numbers, different order
print(bool(s_fwd == s_rev), float(s_fwd - s_rev))

False -0.000152587890625


On a CPU the summation order inside one operation is at least fixed, so
seeded runs repeat; the bitwise agreement of `train_once` above is exactly
that. On a GPU it often is not: kernels built on atomic additions (segment
sums, the scatter-add behind `tf.gather`'s gradient) commit their partial
sums in whatever order threads happen to finish, so two seeded runs on the
*same machine* can differ in the last bits, and after enough training
steps, in the loss curve. `tf.config.experimental.enable_op_determinism()`
is the switch that forbids this: operations with a deterministic
implementation use it (often at some speed cost), operations without one
raise `tf.errors.UnimplementedError` rather than silently varying, and
stateful random operations refuse to run without a seed, since an
operation that seeds itself from entropy is nondeterminism by another
name. Two properties follow from its design. It is meant to be called at
program start, before any operation runs, because it configures kernels as
they are created and nothing already executed is redone. And there is no
call that turns it off short of a fresh process. We can still demonstrate
the seed rule mid-flight by clearing the global seed:

In [6]:
tf.config.experimental.enable_op_determinism()
tf.random.set_seed(None)  # forget the global seed for a moment
try:
    tf.random.normal((2,))
except RuntimeError as e:
    print(str(e).split('.')[0])
tf.random.set_seed(0)  # determinism stays on; reseed and continue

Random ops require a seed to be set when determinism is enabled


This guarantee has a narrow scope. No framework promises bitwise agreement
across releases, platforms, or CPU versus GPU: it holds only for a
pinned machine, library version, and flag configuration. That makes bitwise
reproducibility a *debugging* tool, the setting that lets you bisect
exactly where two runs diverge. The *scientific* goal is statistical
reproducibility: the same conclusions across seeds, reported as a mean and
spread over several runs rather than one fortunate curve. The distinction
mirrors that section: changing dtype changes results in the
last bits by design, and an experimental claim that survives neither a new
seed nor bfloat16 was never a result.

## Hooks: Looking Inside

Keras wraps `call` with a `__call__` too (it handles building, dtype
casting, and masks), but the wrapper exposes no observation point: nothing
can be attached to an unmodified model after the fact, and observing a
black-box model from a library or a checkpoint without touching its code
has no TensorFlow equivalent. Two idioms reach the same measurements with
a little structure. In a *functional* model every
intermediate tensor is a first-class object, so a second `tf.keras.Model`
over the same graph can declare any internal tensor an output: surgery
rather than hooking, sharing all weights and adding no computation. And
where you own the model's code, an overridden `call` can stash or check
whatever it likes as it runs. the figure still draws the right
picture, with one amendment: in TensorFlow the observer cannot stand in
the gap unless the model was built to leave one.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

We reuse the residual stack of
that section, rebuilt compactly:

In [7]:
def residual_block(X, num_hiddens):
    body = tf.keras.Sequential([
        tf.keras.layers.Dense(num_hiddens, activation='relu'),
        tf.keras.layers.Dense(num_hiddens)])
    return X + body(X)

tf.keras.utils.set_random_seed(0)
inputs = tf.keras.Input(shape=(20,))
taps = [tf.keras.layers.Dense(64)(inputs)]  # keep every intermediate tensor
for _ in range(8):
    taps.append(residual_block(taps[-1], 64))
outputs = tf.keras.layers.Dense(10)(taps[-1])
net = tf.keras.Model(inputs, outputs)

### Capturing Activation Statistics

The initialization experiments of that section measured the
standard deviation of activations at every depth. The same measurement
takes a few lines on an unmodified model:

In [8]:
probe = tf.keras.Model(inputs, taps + [outputs])  # same layers, same weights
X = tf.random.normal((256, 20))
names = ['Dense'] + ['ResidualBlock'] * 8 + ['Dense']
for name, out in zip(names, probe(X)):
    print(f'{name:15s} std {float(tf.math.reduce_std(out)):.2f}')

Dense           std 0.67
ResidualBlock   std 0.86
ResidualBlock   std 1.03
ResidualBlock   std 1.26
ResidualBlock   std 1.53
ResidualBlock   std 1.86
ResidualBlock   std 2.23
ResidualBlock   std 2.80
ResidualBlock   std 3.60
Dense           std 3.93


The residual stream's spread grows block by block, since each block adds
its body's output on top of the stream, exactly the depth effect that
motivated the scaled initializations of that section, measured
here without touching the model.

Nothing needs detaching and nothing needs removing: `probe` shares the
original model's layers and weights outright, its outputs are ordinary
tensors with no gradient tape attached, and no observer stays registered
anywhere, because the "hook" is just another model output. The limit is
structural. Surgery needs a functional graph; a subclassed model whose
`call` is imperative Python has no symbolic intermediates to tap. For that
case you override `call` itself, which the next problem gives us a reason
to do.

### A NaN Finder

When a loss becomes NaN at step 40,000, the useful question is which layer
produced the first non-finite value. Check every leaf module's output for
finiteness and the answer arrives in one forward pass. We sabotage a
weight deep in the stack to watch it fire:

In [9]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu'),
            tf.keras.layers.Dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

class Checked(tf.keras.Model):
    def __init__(self, layers):
        super().__init__()
        self.seq, self.first_bad = layers, None  # a plain list is tracked

    def call(self, X):
        for layer in self.seq:
            X = layer(X)
            if self.first_bad is None and not bool(
                    tf.reduce_all(tf.math.is_finite(X))):
                self.first_bad = layer.name
        return X

tf.keras.utils.set_random_seed(0)
checked = Checked([tf.keras.layers.Dense(64)]
                  + [ResidualBlock(64, name=f'block{k}') for k in range(1, 9)]
                  + [tf.keras.layers.Dense(10)])
_ = checked(tf.random.normal((2, 20)))  # build the weights
kernel = checked.seq[3].body.layers[0].kernel
kernel[0, 0].assign(float('nan'))  # sabotage one layer
_ = checked(tf.random.normal((2, 20)))
print('first non-finite output in', checked.first_bad)

first non-finite output in block3


The report names `block3`, the layer we poisoned, rather than leaving you
to bisect with print statements while NaNs propagate through everything
downstream. The check runs on every forward pass until you edit it out of
`call`, the price of building observation into the model rather than
attaching it from outside. For a one-off hunt TensorFlow also ships the
whole idiom as a switch: `tf.debugging.enable_check_numerics()` instruments
every operation and reports the first one to produce an inf or NaN.

### Backward Hooks and Beyond

There are no backward hooks, but gradients do not need them:
`tf.GradientTape` already hands back the gradient of every variable as a
value. To log per-layer gradient norms, catch exploding gradients at their
source, or experiment with per-layer clipping, compute the gradients and
inspect or transform them like any other data. For the specific job of
extracting features from a pretrained backbone, the surgery idiom is also
the production tool:
`tf.keras.Model(backbone.input, backbone.get_layer('avg_pool').output)`
turns any functional backbone into a feature extractor by naming the
tensor you want.

In [10]:
with tf.GradientTape() as tape:
    loss = tf.reduce_mean(net(X) ** 2)
grads = tape.gradient(loss, net.trainable_variables)
print({v.path: float(tf.norm(g))  # block 3's first layer
       for v, g in list(zip(net.trainable_variables, grads))[10:12]})

{'sequential_5/dense_11/kernel': 13.867950439453125, 'sequential_5/dense_11/bias': 2.8245253562927246}


## Summary

Reproducibility is an inventory problem: initialization, dropout,
shuffling, augmentation, and loader workers each draw from some generator,
and a run repeats only if every one of them is seeded.

`tf.keras.utils.set_random_seed` covers Python's `random`, NumPy, and
TensorFlow's global generator in one call; `tf.random.Generator` objects
and `Dataset.shuffle(seed=...)` carry their seeds explicitly. Seeding
makes the program repeatable, not the arithmetic:
`tf.config.experimental.enable_op_determinism()` pins kernel choice too,
raising on operations that cannot comply, and wants to be the first line
of the program.

Bitwise agreement is a debugging tool;
conclusions that hold across seeds are the scientific goal.

For looking inside a model there is no hook to attach: declare the tensors
you want as extra outputs of a functional model, or override `call` where
you own the code, and read gradients as values from `tf.GradientTape`.

## Exercises

1. Extend `train_once` to load data through your framework's parallel data
   loader (e.g., PyTorch's `DataLoader`, with `num_workers=4`). Give the
   dataset its own `np.random.default_rng()` object and use it to add noise
   as each example loads. On a process-based loader, check whether workers
   copied the same generator state. Using the per-worker seeding hook this
   section introduced for your framework, replace the dataset's stashed
   generator with one derived from the worker's own seed, then verify
   agreement across two runs.
2. Write a forward hook (or your framework's equivalent introduced in this
   section) that counts multiply-accumulate operations for every linear
   layer from the shapes of its input and weight, and use it to report
   per-layer and total FLOPs for the residual stack above. Check the total
   against a hand count.
3. Using a backward hook (or your framework's equivalent introduced in this
   section), record the norm of each block's output gradient. After the
   backward pass, compare these activation-gradient norms with the
   parameter-gradient norms in the same block. Which one first reveals an
   exploding backward signal?
4. A forward hook that returns a value *replaces* the module's output. Use
   one to zero out the output of a single residual block's body (turning
   the block into the identity) and measure how the network's output
   changes. Which block matters most, and how would you find out with one
   loop?

5. Rebuild the activation-statistics table two more ways: with a
   `Checked`-style `call` override that stashes `tf.math.reduce_std` of
   every layer's output, and on a model you did not write, a
   `tf.keras.applications` backbone, by naming layers with `get_layer`.
   Compare the three contracts you now know, functional surgery, `call`
   overrides, and PyTorch-style mutable hooks: which requires a symbolic
   graph, which requires owning the model's code, and which black-box
   models does each admit?